In [5]:
%pip install pandas numpy matplotlib seaborn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import pandas as pd
import numpy as np
import random
np.random.seed(42)
random.seed(42)
try: 
    df_expenses_raw = pd.read_csv('Expenses_clean.csv')
    df_income_raw = pd.read_csv('Income_clean.csv')
    df_budget_raw = pd.read_csv('budget_data.csv')
    df_tracker_raw = pd.read_csv('personal_finance_tracker_dataset.csv')
    df_8000_raw = pd.read_csv('personal_finance_dataset_8000_extended.csv')
except FileNotFoundError as e:
    print(f" Thiếu file trên hệ thống. Vui lòng cập nhật file vào! {e}")
    raise
# GÂY NHIỄU CHO CÁC FILE DỮ LIỆU
#1. Gây nhiễu file Expenses - Income - Budget
# Tạo ô trống ngẫu nhiên ở cột số tiền - Amount và danh mục - Category
mask_exp_amt = np.random.rand(len(df_expenses_raw)) < 0.05
mask_exp_cat = np.random.rand(len(df_expenses_raw)) < 0.04
mask_inc_amt = np.random.rand(len(df_income_raw)) < 0.05
mask_inc_cat = np.random.rand(len(df_income_raw)) < 0.05
df_budget_raw.loc[np.random.rand(len(df_budget_raw)) < 0.03, 'amount'] = np.nan
df_expenses_raw.loc[mask_exp_amt, 'amount'] = np.nan
df_expenses_raw.loc[mask_exp_cat, 'category'] = np.nan
df_income_raw.loc[mask_inc_amt, 'amount'] = np.nan
df_income_raw.loc[mask_inc_cat, 'category'] = np.nan
# Gây lỗi định dạng ngày tháng
df_expenses_raw['date_time'] = (df_expenses_raw['date_time'].astype(str).apply(lambda x: "ERR_DATE_FORMAT_99" if random.random() < 0.03 else x)
)
df_income_raw['date_time'] = df_income_raw['date_time'].astype(str).apply(lambda x: "ERR_DATE_FORMAT_99" if random.random() < 0.04 else x
)
#Lỗi Trùng lặp
ty_le_trung_exp = 0.10  
so_dong_trung_exp = int(len(df_expenses_raw) * ty_le_trung_exp)
df_dup_exp = df_expenses_raw.sample(n = so_dong_trung_exp, random_state = 12)
df_exp_noisy = pd.concat([df_expenses_raw, df_dup_exp], ignore_index = True)
df_exp_noisy = df_exp_noisy.sample(frac = 1, random_state = 12).reset_index(drop = True)
#2. Gây nhiễu cho file 8000_extended
# Tạo giá trị trống khuyết thiếu ngẫu nhiên ở cột số tiền - Amount và danh mục - Category
mask_amt = np.random.rand(len(df_8000_raw)) < 0.05
mask_cat = np.random.rand(len(df_8000_raw)) < 0.04
df_8000_raw['Amount'] = df_8000_raw['Amount'].mask(mask_amt)
df_8000_raw['Category'] = df_8000_raw['Category'].mask(mask_cat)
# Tạo thêm giao dịch bất thường với dòng tiền lớn hơn thực tế
outlier_idx = np.random.choice(df_8000_raw.index, size = 50, replace = False)
df_8000_raw.loc[outlier_idx, 'Amount'] = df_8000_raw.loc[outlier_idx, 'Amount'] * 100
# Tạo lỗi kí tự và chính tả ở danh mục
df_8000_raw['Category'] = df_8000_raw['Category'].replace({'Travel': 'Travell_err', 'Grocery': 'Grocerie$', 'Bills': 'Biills'})
#3. Gây nhiễu cho file Trakcer_dataset
# Tạo giá trị trống hệ thống ở các đặc trưng cốt lõi
for col in ['monthly_income', 'credit_score', 'debt_to_income_ratio', 'financial_stress_level']:
    mask = np.random.rand(len(df_tracker_raw)) < 0.06
    df_tracker_raw[col] = df_tracker_raw[col].mask(mask)
# Giá trị vô lí cho điểm tín dụng
bad_idx = np.random.choice(df_tracker_raw.index, size = 15, replace = False)
df_tracker_raw.loc[bad_idx, 'credit_score'] = -999.0
# XUẤT CÁC FILE NHIỄU
df_income_raw.to_csv("Income_Noisy.csv", index = False)
df_8000_raw.to_csv("8000_Extended_Noisy.csv", index = False)
df_budget_raw.to_csv("Budget_Noisy.csv", index = False)
df_tracker_raw.to_csv("Tracker_Noisy.csv", index = False)
df_expenses_raw.to_csv("Expenses_Noisy.csv", index = False)  
